# STEP-003 — Kaggle `DS0001` Audit, Verification & Auto-Push to GitHub

**Project:** AI-Generated Face Detection for Digital Identity Verification (e-KYC) and Fraud Prevention (`GROUP 08`)  
**Owner:** Member 1 — Md Assaduzzaman (`0242310005101233`)  
**Dataset:** `kaggle:isthisdeception/ds0001-artifact-face-subset` (ArtiFact Face Subset, MIT License)  
**Single Source of Truth:** `https://github.com/isthisdeception/computer-vission-course-work`

### One-Click Kaggle Checklist (Before clicking **Run All**)
1. **Add Input → Datasets:** Attach `isthisdeception/ds0001-artifact-face-subset`
2. **Session Options → Internet:** Turn **ON** (needed to push outputs back to GitHub)
3. **Add-ons → Secrets:** Toggle **ON** your GitHub access token secret (`GITHUB_TOKEN` — also auto-detects `GH_TOKEN` / `GITHUB_PAT` / `GIT_TOKEN`)
4. Click **Run All** — when finished, `ds0001_index.csv`, `ds0001_audit_summary.json`, `ds0001_annotation_examples.png`, `figure_registry.csv`, and `PROGRESS_TRACKER.md` are automatically verified and pushed back to `main`!


In [ ]:
import hashlib
import json
import os
import shutil
import subprocess
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

SEED = 42
REPO_URL = "https://github.com/isthisdeception/computer-vission-course-work.git"
BRANCH = "main"
GIT_USER_NAME = "Assaduzzaman"
GIT_USER_EMAIL = "md.assaduzzaman99@gmail.com"
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

OUTPUT_DIR = Path("/kaggle/working/step003_outputs") if Path("/kaggle/working").exists() else Path("02_data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Output directory:", OUTPUT_DIR.resolve())


In [ ]:
# 1. Auto-detect DS0001 root directory on Kaggle (supports both classic and datasets/ mount layouts)
CANDIDATE_ROOTS = [
    Path("/kaggle/input/datasets/isthisdeception/ds0001-artifact-face-subset"),
    Path("/kaggle/input/ds0001-artifact-face-subset"),
]

MOUNT_ROOT = None
for cand in CANDIDATE_ROOTS:
    if cand.exists():
        MOUNT_ROOT = cand
        break

if MOUNT_ROOT is None:
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        matches = [p for p in kaggle_input.rglob("*ds0001*") if p.is_dir()]
        if matches:
            MOUNT_ROOT = matches[0]

assert MOUNT_ROOT is not None, (
    "DS0001 not found! Attach Kaggle dataset 'isthisdeception/ds0001-artifact-face-subset' to this notebook."
)

# Locate the actual directory containing real/ and fake/ image folders
DATA_ROOT = MOUNT_ROOT
for p in [MOUNT_ROOT, *MOUNT_ROOT.rglob("*")]:
    if p.is_dir() and (p / "real").is_dir() and (p / "fake").is_dir():
        DATA_ROOT = p
        break

METADATA_CSVS = sorted(MOUNT_ROOT.rglob("metadata.csv"))
print("MOUNT_ROOT   :", MOUNT_ROOT)
print("DATA_ROOT    :", DATA_ROOT)
print("METADATA_CSVS:", [str(p) for p in METADATA_CSVS])


In [ ]:
# 2. Load metadata.csv mapping -> {key: (label_name, generator)}
def norm_path(p: str) -> str:
    return str(p).replace("\\", "/").strip().lstrip("./")

meta_lookup = {}
for csv_path in METADATA_CSVS:
    df_meta = pd.read_csv(csv_path)
    print(f"Loaded {csv_path.name} ({len(df_meta)} rows) | columns: {list(df_meta.columns)}")
    cols = {c.lower(): c for c in df_meta.columns}
    fn_col = cols.get("filename") or cols.get("file_name") or cols.get("image")
    op_col = cols.get("output_path") or cols.get("rel_path") or cols.get("path")
    lbl_col = cols.get("label") or cols.get("class")
    gen_col = cols.get("generator") or cols.get("source") or cols.get("model")

    for row in df_meta.itertuples(index=False):
        row_dict = row._asdict()
        raw_lbl = str(row_dict.get(lbl_col, "")).strip().lower() if lbl_col else ""
        if raw_lbl in ("0", "real", "pristine"):
            label_name = "real"
        elif raw_lbl in ("1", "fake", "ai", "synthetic"):
            label_name = "fake"
        else:
            label_name = raw_lbl or "unknown"

        raw_gen = str(row_dict.get(gen_col, "")).strip().lower() if gen_col else ""
        if label_name == "real" and (not raw_gen or raw_gen in ("nan", "none", "null")):
            generator = "none"
        else:
            generator = raw_gen if raw_gen and raw_gen != "nan" else ("none" if label_name == "real" else "unknown")

        entry = (label_name, generator)
        if fn_col and pd.notna(row_dict.get(fn_col)):
            fn = norm_path(row_dict[fn_col])
            meta_lookup[fn] = entry
            meta_lookup[Path(fn).name] = entry
            meta_lookup[f"real/{Path(fn).name}"] = entry
            meta_lookup[f"fake/{Path(fn).name}"] = entry
        if op_col and pd.notna(row_dict.get(op_col)):
            op = norm_path(row_dict[op_col])
            meta_lookup[op] = entry
            meta_lookup[Path(op).name] = entry
            parts = op.split("/")
            if len(parts) >= 2:
                meta_lookup["/".join(parts[-2:])] = entry

print("Total lookup keys indexed:", len(meta_lookup))


In [ ]:
# 3. Enumerate all image files under DATA_ROOT and audit readability, dimensions & MD5
all_files = sorted(
    p for p in DATA_ROOT.rglob("*")
    if p.is_file()
    and p.suffix.lower() in IMAGE_EXTS
    and "__MACOSX" not in p.parts
    and not p.name.startswith("._")
)
print(f"Total image files discovered: {len(all_files):,}")


def audit_single_image(img_path: Path):
    rel_path = norm_path(img_path.relative_to(DATA_ROOT))
    base = img_path.name
    parent_key = "/".join(rel_path.split("/")[-2:])

    label_name, generator = "unknown", "unknown"
    for key in (rel_path, parent_key, base):
        if key in meta_lookup:
            label_name, generator = meta_lookup[key]
            break
    if label_name == "unknown":
        low = f"/{rel_path.lower()}/"
        if "/real/" in low:
            label_name, generator = "real", "none"
        elif "/fake/" in low:
            label_name, generator = "fake", "unknown"

    label_int = 0 if label_name == "real" else (1 if label_name == "fake" else -1)

    try:
        raw_bytes = img_path.read_bytes()
        if len(raw_bytes) == 0:
            return {"ok": False, "rel_path": rel_path, "error": "empty_file"}
        md5_hash = hashlib.md5(raw_bytes).hexdigest()

        with Image.open(img_path) as im:
            im.verify()
        with Image.open(img_path) as im:
            width, height = im.size
            mode = im.mode
            fmt = im.format or img_path.suffix.lstrip(".").upper()

        return {
            "ok": True,
            "rel_path": rel_path,
            "label": label_int,
            "label_name": label_name,
            "generator": generator,
            "width": int(width),
            "height": int(height),
            "mode": str(mode),
            "format": str(fmt),
            "file_ext": img_path.suffix.lower(),
            "md5_hash": md5_hash,
        }
    except Exception as exc:
        return {"ok": False, "rel_path": rel_path, "error": str(exc)}


records = []
corrupt = []
with ThreadPoolExecutor(max_workers=8) as ex:
    for res in tqdm(ex.map(audit_single_image, all_files), total=len(all_files), desc="Auditing DS0001"):
        if res["ok"]:
            records.append(res)
        else:
            corrupt.append(res)

print(f"Readable images: {len(records):,} | Corrupt images: {len(corrupt):,}")


In [ ]:
# 4. Build canonical ds0001_index.csv and ds0001_audit_summary.json
df_idx = pd.DataFrame(records).sort_values("rel_path").reset_index(drop=True)
df_idx.insert(0, "image_id", [f"DS0001_{i:06d}" for i in range(1, len(df_idx) + 1)])

hash_counts = df_idx["md5_hash"].value_counts()
dup_hash_groups = int((hash_counts > 1).sum())
dup_extra_files = int((hash_counts[hash_counts > 1] - 1).sum())

audit_summary = {
    "dataset_id": "DS0001",
    "dataset_name": "ArtiFact Face Subset (50k)",
    "kaggle_slug": "isthisdeception/ds0001-artifact-face-subset",
    "license": "MIT",
    "built_utc": datetime.now(timezone.utc).isoformat(),
    "total_scanned": len(all_files),
    "readable_count": len(df_idx),
    "corrupt_count": len(corrupt),
    "corrupt_rate": round(len(corrupt) / max(len(all_files), 1), 6),
    "class_counts": df_idx["label_name"].value_counts().to_dict(),
    "generator_counts": df_idx["generator"].value_counts().to_dict(),
    "num_fake_generators": int(df_idx.loc[df_idx["label"] == 1, "generator"].nunique()),
    "width_min": int(df_idx["width"].min()),
    "width_max": int(df_idx["width"].max()),
    "height_min": int(df_idx["height"].min()),
    "height_max": int(df_idx["height"].max()),
    "format_histogram": df_idx["format"].value_counts().to_dict(),
    "mode_histogram": df_idx["mode"].value_counts().to_dict(),
    "duplicate_md5_groups": dup_hash_groups,
    "duplicate_md5_extra_files": dup_extra_files,
}

CANONICAL_COLS = [
    "image_id", "rel_path", "label", "label_name", "generator",
    "width", "height", "file_ext", "md5_hash",
]
index_csv_path = OUTPUT_DIR / "ds0001_index.csv"
summary_json_path = OUTPUT_DIR / "ds0001_audit_summary.json"

df_idx[CANONICAL_COLS].to_csv(index_csv_path, index=False)
summary_json_path.write_text(json.dumps(audit_summary, indent=2), encoding="utf-8")

print("Saved:", index_csv_path)
print("Saved:", summary_json_path)
print(json.dumps(audit_summary, indent=2))

assert len(df_idx) == 50000, f"Expected 50000 images, found {len(df_idx)}"
assert len(corrupt) == 0, f"Found {len(corrupt)} corrupt images!"
assert audit_summary["class_counts"].get("real") == 25000
assert audit_summary["class_counts"].get("fake") == 25000
assert audit_summary["num_fake_generators"] == 13
print("\n[PASS] All DS0001 integrity assertions passed!")


In [ ]:
# 5. Preview annotated examples (4 Real + 4 Diverse AI Generators) for Milestone 2 evidence
real_samples = df_idx[df_idx["label"] == 0].sample(4, random_state=SEED)
fake_gens = ["stable_diffusion", "stylegan2", "stylegan3", "taming_transformer"]
fake_samples = pd.concat([
    df_idx[df_idx["generator"] == g].sample(1, random_state=SEED)
    for g in fake_gens if (df_idx["generator"] == g).any()
])
preview_df = pd.concat([real_samples, fake_samples], ignore_index=True)

fig, axes = plt.subplots(2, 4, figsize=(12, 6.5), dpi=200)
for ax, row in zip(axes.flat, preview_df.itertuples(index=False)):
    im = Image.open(DATA_ROOT / row.rel_path)
    ax.imshow(im)
    color = "#1b7f3b" if row.label == 0 else "#b42318"
    ax.set_title(
        f"{row.label_name.upper()} | {row.generator}\n{row.image_id} ({row.width}x{row.height})",
        fontsize=8.5,
        color=color,
        fontweight="bold",
    )
    ax.axis("off")

plt.suptitle("DS0001 (ArtiFact Face Subset) — Verified Annotated Sample Preview", fontsize=12, fontweight="bold")
plt.tight_layout()
preview_path = OUTPUT_DIR / "ds0001_annotation_examples.png"
plt.savefig(preview_path, dpi=200, bbox_inches="tight")
plt.show()
print("Saved annotation preview:", preview_path)
preview_df[CANONICAL_COLS[:7]]


In [ ]:
# 6. Auto-Verify & Push All STEP-003 Outputs Directly Back to GitHub (Secret-Safe)
SECRET_NAMES = ("GITHUB_TOKEN", "GH_TOKEN", "GITHUB_PAT", "GIT_TOKEN", "GITHUB_ACCESS_TOKEN", "github_token")


def _load_github_token():
    try:
        from kaggle_secrets import UserSecretsClient
        client = UserSecretsClient()
        for name in SECRET_NAMES:
            try:
                val = client.get_secret(name)
                if val and val.strip():
                    return val.strip(), f"Kaggle Secret '{name}'"
            except Exception:
                continue
    except ImportError:
        pass
    for name in SECRET_NAMES:
        val = os.environ.get(name, "").strip()
        if val:
            return val, f"env '{name}'"
    raise RuntimeError(
        "GitHub token not found! Enable your token under Kaggle Add-ons -> Secrets (e.g., GITHUB_TOKEN)."
    )


def _run_masked(cmd, cwd=None, secret=None):
    res = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True, check=False)
    mask = (lambda s: s.replace(secret, "***")) if secret else (lambda s: s)
    if res.stdout.strip():
        print(mask(res.stdout.strip()))
    if res.returncode != 0:
        if res.stderr.strip():
            print(mask(res.stderr.strip()))
        raise RuntimeError(f"Git command failed (rc={res.returncode}): {mask(' '.join(cmd))}")
    return mask(res.stdout.strip())


token, token_src = _load_github_token()
print(f"[GitHub Sync] Token loaded from {token_src} (masked).")

repo_dir = Path("/kaggle/working/_repo_sync")
if repo_dir.exists():
    shutil.rmtree(repo_dir)

auth_url = REPO_URL.replace("https://", f"https://x-access-token:{token}@", 1)
_run_masked(["git", "clone", "--depth", "1", "--branch", BRANCH, auth_url, str(repo_dir)], secret=token)
_run_masked(["git", "remote", "set-url", "origin", REPO_URL], cwd=repo_dir, secret=token)
_run_masked(["git", "config", "user.name", GIT_USER_NAME], cwd=repo_dir, secret=token)
_run_masked(["git", "config", "user.email", GIT_USER_EMAIL], cwd=repo_dir, secret=token)

# Copy outputs to exact canonical paths
copy_map = {
    index_csv_path: "02_data/ds0001_index.csv",
    summary_json_path: "02_data/ds0001_audit_summary.json",
    preview_path: "06_evaluation/figures/eda/ds0001_annotation_examples.png",
}
for src_p, rel_dst in copy_map.items():
    dst_p = repo_dir / rel_dst
    dst_p.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src_p, dst_p)
    print(f"[GitHub Sync] Staged -> {rel_dst}")

# Run repository's official verify_index.py check inside the cloned repo
_run_masked(["python", "02_data/verify_index.py"], cwd=repo_dir, secret=token)

# Register annotation preview figure in figure_registry.csv (idempotent)
fig_reg_path = repo_dir / "06_evaluation/figure_registry.csv"
fig_df = pd.read_csv(fig_reg_path)
if "FIG00" not in set(fig_df["fig_id"].astype(str)):
    new_row = pd.DataFrame([{
        "fig_id": "FIG00",
        "filename": "06_evaluation/figures/eda/ds0001_annotation_examples.png",
        "step_id": "STEP-003",
        "report_section": "Section 6",
        "caption": "DS0001 (ArtiFact Face Subset) verified annotated sample preview (4 Real vs. 4 AI generators)",
        "source_script": "04_notebooks/01_ds0001_audit_and_index.ipynb",
    }])
    pd.concat([fig_df, new_row], ignore_index=True).to_csv(fig_reg_path, index=False)

# Mark STEP-003 DONE in PROGRESS_TRACKER.md
tracker_path = repo_dir / "PROGRESS_TRACKER.md"
tracker_text = tracker_path.read_text(encoding="utf-8")
today_str = datetime.now(timezone.utc).strftime("%Y-%m-%d")
updated_lines = []
for line in tracker_text.splitlines():
    if line.startswith("| `STEP-003` |"):
        line = (
            f"| `STEP-003` | M2 (Due 22 Nov — Catch-up) | Kaggle `DS0001` Verification, License & Master Index CSV "
            f"| M1 Md Assaduzzaman | Nov 26 | `DONE` | `ds0001_index.csv` (50k, 0 corrupt) pushed from Kaggle ({today_str}) |"
        )
    updated_lines.append(line)
tracker_path.write_text("\n".join(updated_lines) + "\n", encoding="utf-8")

# Commit and push straight back to main
_run_masked(["git", "add", "-A"], cwd=repo_dir, secret=token)
status_out = _run_masked(["git", "status", "--porcelain"], cwd=repo_dir, secret=token)
if status_out.strip():
    _run_masked(
        ["git", "commit", "-m", "data(step-003): push verified DS0001 index, audit summary, and sample grid from Kaggle"],
        cwd=repo_dir,
        secret=token,
    )
    _run_masked(["git", "push", auth_url, f"HEAD:refs/heads/{BRANCH}"], cwd=repo_dir, secret=token)
    sha = _run_masked(["git", "rev-parse", "--short", "HEAD"], cwd=repo_dir, secret=token)
    print(f"\n[SUCCESS] STEP-003 outputs pushed to GitHub main ({sha})!")
else:
    print("\n[INFO] Nothing new to commit — GitHub main already has identical outputs.")

print("\n=== Ready-to-Paste Google Sheet Milestone 2 Evidence URLs ===")
print("Evidence URL 1: https://github.com/isthisdeception/computer-vission-course-work/blob/main/02_data/dataset_card.md")
print("Evidence URL 2: https://github.com/isthisdeception/computer-vission-course-work/blob/main/06_evaluation/figures/eda/ds0001_annotation_examples.png")
